# **Data Visualization with Maps - Folium**

"Folium builds on the data wrangling strengths of the Python ecosystem and the mapping strengths of the Leaflet.js library. Manipulate your data in Python, then visualize it in a Leaflet map via Folium." - Source: [Folium](https://python-visualization.github.io/folium/latest/)

For Folium maps, there are two main components.
1. The map object
2. The markers to be put onto the map

In [ ]:
# Import the necessary libraries
import pandas as pd
import folium

In [ ]:
# Reading and cleaning the data used for visualization
crime = pd.read_csv('crime-new.csv', parse_dates=['occurrencedate'])
crime = crime[crime['occurrenceyear']>=2014]
crime.head()

,X,Y,Index_,event_unique_id,occurrencedate,reporteddate,premisetype,ucr_code,ucr_ext,offence,...,occurrencedayofyear,occurrencedayofweek,occurrencehour,MCI,Division,Hood_ID,Neighbourhood,Lat,Long,FID
0,-79.520401,43.768829,14601,GO-20142775022,2014-08-25 04:00:00+00:00,2014-08-25T04:00:00.000Z,Outside,1430,100,Assault,...,237.0,Monday,18,Assault,D31,24,Black Creek (24),43.768829,-79.520401,14001
1,-79.580856,43.642574,14602,GO-20142870874,2014-08-25 04:00:00+00:00,2014-09-08T04:00:00.000Z,House,2120,220,B&E W'Intent,...,237.0,Monday,9,Break and Enter,D22,11,Eringate-Centennial-West Deane (11),43.642574,-79.580856,14002
2,-79.260445,43.762909,14603,GO-20142802386,2014-08-25 04:00:00+00:00,2014-08-29T04:00:00.000Z,House,1430,100,Assault,...,237.0,Monday,11,Assault,D41,127,Bendale (127),43.762909,-79.260445,14003
3,-79.367546,43.663208,14604,GO-20142777955,2014-08-25 04:00:00+00:00,2014-08-26T04:00:00.000Z,Commercial,2120,200,B&E,...,237.0,Monday,17,Break and Enter,D51,71,Cabbagetown-South St.James Town (71),43.663208,-79.367546,14004
4,-79.231758,43.776440,14605,GO-20142778699,2014-08-25 04:00:00+00:00,2014-08-26T04:00:00.000Z,Other,2120,200,B&E,...,237.0,Monday,16,Break and Enter,D43,137,Woburn (137),43.776440,-79.231758,14005


## **Prepare the Data**

In [ ]:
# Creating the MCI dataframe
mcis = crime.groupby('Neighbourhood')['MCI'].count()
mcis

Neighbourhood
Agincourt North (129)                  752
Agincourt South-Malvern West (128)    1027
Alderwood (20)                         365
Annex (95)                            2270
Banbury-Don Mills (42)                 735
                                      ... 
Wychwood (94)                          485
Yonge-Eglinton (100)                   255
Yonge-St.Clair (97)                    193
York University Heights (27)          2515
Yorkdale-Glen Park (31)               1197
Name: MCI, Length: 140, dtype: int64

In [ ]:
# Creating the map_data dataframe
map_data = crime[['Neighbourhood', 'Lat', 'Long']]\
            .drop_duplicates('Neighbourhood').set_index('Neighbourhood') \
            .join(mcis, how='inner')
map_data

,Lat,Long,MCI
Neighbourhood,,,
Black Creek (24),43.768829,-79.520401,1620
Eringate-Centennial-West Deane (11),43.642574,-79.580856,568
Bendale (127),43.762909,-79.260445,1799
Cabbagetown-South St.James Town (71),43.663208,-79.367546,877
Woburn (137),43.776440,-79.231758,2519
...,...,...,...
High Park-Swansea (87),43.640636,-79.449158,526
Englemount-Lawrence (32),43.729061,-79.445992,699
Corso Italia-Davenport (92),43.676472,-79.449829,631


## **Basic Map**

A very basic map can be created using `folium.Map()`.

It requires two basic parameters:
- location is the center of the map (It takes an array of latitude and longitude)
- zoom_start is the starting zoom level of the map. It takes an integer.

In [ ]:
# Displaying a basic map
folium.Map(
    location=[43.666550, -79.385261],
    zoom_start=11
)

## **Adding Markers**

Markers can be created by using `folium.Marker()`. It requires the latitude and longitude as an array.

To display the marker, it needs to be added to the map using `.add_to()` method.

In [ ]:
# Creating the map object
m = folium.Map(
    location = [43.666550, -79.385261],
    zoom_start = 10
)

# Getting the latitude and longitude of the first row of map_data dataset
row1 = map_data.iloc[0]
lat = row1.Lat
long = row1.Long

# Adding the marker to the map
folium.Marker([lat,long]).add_to(m)

# Displaying map
m

### **Popups**
Popups are used to show some information about the marker when it is clicked. It can be added with `popup` parameter.

In [ ]:
# Creating the map object
m = folium.Map(
    location = [43.666550, -79.385261],
    zoom_start = 10
)

# Getting the latitude and longitude of the first row of map_data dataset
lat = row1.Lat
long = row1.Long

# Adding the marker to the map
folium.Marker([lat,long], popup = row1.MCI).add_to(m)

# Displaying map
m

### **Adding Tooltips**

Tooltips can also give more information about the marker, but it works when the cursor hovers over the marker. It can be added using the `tooltip` parameter.

In [ ]:
# Creating the map object
m = folium.Map(
    location = [43.666550, -79.385261],
    zoom_start = 10
)

# Getting the latitude and longitude of the first row of map_data dataset
lat = row1.Lat
long = row1.Long

# Adding the marker to the map
folium.Marker([lat,long], tooltip = row1.MCI).add_to(m)

# Displaying map
m

## **Putting it all Together**

Putting everything together, the entire crime dataset can be displayed on the map using a for loop.

In [ ]:
m = folium.Map(
    location=[43.666550, -79.385261],
    zoom_start=11
)

for x in map_data.iterrows():
    name = " ".join(x[0].split()[:2])
    count = int(x[1].MCI)
    folium.Marker([x[1].Lat, x[1].Long],
                 popup = count,
                 tooltip = name).add_to(m)
m